# ✅ SOLUTION NOTEBOOK 1: LangSmith Tracing — Multi-Tool Agent

---

> ⚠️ **This is the reference solution.** Attempt the practice notebook first before looking here!

---


In [ ]:
!pip install langchain langchain-openai langsmith langchain-community

## ✅ Step 1 Solution — Environment Variables

In [1]:
import os
from dotenv import load_dotenv
# # All four required environment variables
# os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"          # ← replace with your key
# os.environ["LANGCHAIN_API_KEY"] = "your-langchain-api-key-here"    # ← replace with your key
os.environ["LANGCHAIN_TRACING_V2"] = "true"                        # must be string "true"
os.environ["LANGCHAIN_PROJECT"] = "Activity_1"       # project name in LangSmith

print("Project:", os.environ.get("LANGCHAIN_PROJECT"))
print("Tracing:", os.environ.get("LANGCHAIN_TRACING_V2"))

Project: Activity_1
Tracing: true


## ✅ Step 2 Solution — LLM Setup

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langsmith import traceable

# Create the LLM — gpt-4o-mini, deterministic output with temperature=0
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("LLM ready:", llm is not None)

LLM ready: True


## ✅ Step 3 Solution — Two Traceable Tools

**Key insight:** Using `run_type="tool"` marks these as tool spans in LangSmith,
so they appear nested *under* the parent chain `run_agent` in the dashboard.

In [3]:
@traceable(run_type="tool", name="WeatherTool")
def weather_tool(query: str) -> str:
    """
    Simulated weather tool.
    In a real agent this would call a weather API.
    """
    return "It is sunny and 25°C in your location today."


@traceable(run_type="tool", name="JokeTool")
def joke_tool(query: str) -> str:
    """
    Simulated joke tool.
    In a real agent this would call a jokes API or dedicated LLM prompt.
    """
    return "Why don't scientists trust atoms? Because they make up everything!"


print("Weather tool:", weather_tool("test"))
print("Joke tool:", joke_tool("test"))

Weather tool: It is sunny and 25°C in your location today.
Joke tool: Why don't scientists trust atoms? Because they make up everything!


## ✅ Step 4 Solution — Main Agent with Routing

**Key insight:** Calling `weather_tool()` or `joke_tool()` *inside* a `@traceable` function
automatically creates a parent-child trace relationship. LangSmith shows these as nested spans.

In [4]:
@traceable(run_type="chain", name="MultiToolAgent")
def run_agent(query: str) -> str:
    """
    Routes the query to the appropriate tool or falls back to the LLM.
    
    Routing logic:
      - 'weather' in query → WeatherTool
      - 'joke' in query    → JokeTool
      - otherwise          → LLM via ChatPromptTemplate
    """
    query_lower = query.lower()

    if "weather" in query_lower:
        # Tool path — creates a child span in LangSmith
        return weather_tool(query)

    elif "joke" in query_lower:
        # Tool path — creates a child span in LangSmith
        return joke_tool(query)

    else:
        # LLM fallback — the ChatPromptTemplate + LLM chain also creates child spans
        prompt = ChatPromptTemplate.from_template(
            "You are a helpful assistant. Answer concisely.\n\nQuestion: {question}"
        )
        chain = prompt | llm
        result = chain.invoke({"question": query})
        return result.content

## ✅ Step 5 Solution — Run with All Four Query Types

In [5]:
test_queries = [
    "What is the weather like today?",              # → WeatherTool
    "Tell me a joke please",                         # → JokeTool
    "What is the capital of France?",                # → LLM fallback
    "Explain the difference between RAM and ROM",    # → LLM fallback (learner's own query)
]

for q in test_queries:
    print(f"\n🔍 Query: {q}")
    result = run_agent(q)
    print(f"💬 Answer: {result}")
    print("-" * 70)


🔍 Query: What is the weather like today?
💬 Answer: It is sunny and 25°C in your location today.
----------------------------------------------------------------------

🔍 Query: Tell me a joke please
💬 Answer: Why don't scientists trust atoms? Because they make up everything!
----------------------------------------------------------------------

🔍 Query: What is the capital of France?
💬 Answer: The capital of France is Paris.
----------------------------------------------------------------------

🔍 Query: Explain the difference between RAM and ROM
💬 Answer: RAM (Random Access Memory) is a type of volatile memory used for temporary data storage while a computer is running, allowing for quick read and write access. It loses its data when the power is turned off. 

ROM (Read-Only Memory), on the other hand, is non-volatile memory that permanently stores firmware and system instructions. It retains its data even when the power is off and is typically not writable during normal operation.


---
## 📝 Sample Answers for Step 6 Observations

**Q1 — Number of traces:** 4 traces (one per query run)

**Q2 — Slowest trace:** The LLM fallback traces will be slowest (~500ms–2s) because they make a real network call to the OpenAI API. Tool traces are near-instant (<5ms) since they return hardcoded strings.

**Q3 — Parent-child nesting:** Yes. Click on a `MultiToolAgent` trace in LangSmith. You will see a tree with `MultiToolAgent` at the root and `WeatherTool` or `JokeTool` as child spans underneath it. For LLM traces, you'll see the `ChatPromptTemplate` and `ChatOpenAI` as children.

**Q4 — LLM trace inputs/outputs:** The LLM trace shows the full prompt after template substitution (inputs) and the model's response text (outputs). You can also see token counts and model name.

---
## ✅ Bonus Solution — MathTool

In [6]:
@traceable(run_type="tool", name="MathTool")
def math_tool(query: str) -> str:
    """
    Extracts two numbers from the query and returns their sum.
    Example: 'calculate 45 and 72' → 'The sum of 45.0 and 72.0 is 117.0'
    """
    # Extract all numeric tokens from the query
    tokens = query.replace("and", " ").split()
    numbers = []
    for token in tokens:
        try:
            numbers.append(float(token))
        except ValueError:
            pass
    
    if len(numbers) < 2:
        raise ValueError("MathTool needs at least two numbers in the query")
    
    total = numbers[0] + numbers[1]
    return f"The sum of {numbers[0]} and {numbers[1]} is {total}"


@traceable(run_type="chain", name="MultiToolAgentV2")
def run_agent_v2(query: str) -> str:
    """Extended agent with MathTool."""
    query_lower = query.lower()

    if "weather" in query_lower:
        return weather_tool(query)
    elif "joke" in query_lower:
        return joke_tool(query)
    elif "calculate" in query_lower or "compute" in query_lower:
        return math_tool(query)
    else:
        prompt = ChatPromptTemplate.from_template(
            "You are a helpful assistant. Answer concisely.\n\nQuestion: {question}"
        )
        chain = prompt | llm
        result = chain.invoke({"question": query})
        return result.content


# Test MathTool
print(run_agent_v2("calculate 45 and 72"))
print(run_agent_v2("compute 100 and 250"))

The sum of 45.0 and 72.0 is 117.0
The sum of 100.0 and 250.0 is 350.0
